Basic workflow for making a "surrogate" model that learns the unpolarized cross-section as function of $x_{\text{B}}$, $t$, $Q^{2}$, and $\phi$.

In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import regularizers
from sklearn.model_selection import train_test_split

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

In [ ]:
test_dataframe = pd.read_csv('../data/small_dset.csv')

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_beam_unp_target_xsec"]]

In [ ]:
x_data.head()

In [ ]:
y_data.head()

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = 0.1, shuffle = True)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True)

In [ ]:
len(x_training)

In [ ]:
len(x_validation)

In [ ]:
len(x_testing)

Loss:

In [ ]:
def cross_section_loss(true_values, predicted_values):

    # extract predictions:
    cross_section_prediction = predicted_values["cross-section"]

    # extract true value:
    true_cross_section = true_values
    
    # compute cross-section residuals:
    residuals_cross_section = true_cross_section - cross_section_prediction

    # compute the MSE:
    mean_squared_error = tf.reduce_mean(tf.square(residuals_cross_section))

    # return it:
    return mean_squared_error

Architecture:

In [ ]:
class CrossSectionSurrogateModel(tf.keras.Model):

    def __init__(self):
        super().__init__()

        initializer = tf.keras.initializers.GlorotNormal(seed = None)

        self.dense_layer_1 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_2 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_3 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")

        # linear activation is default activation if `activation` key is not specified: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
        self.cross_section_output = tf.keras.layers.Dense(1, activation = "linear", name = "cross-section")

    def call(self, inputs, training = False):

        # hidden layer computation:
        hidden_layer = self.dense_layer_1(inputs)
        hidden_layer = self.dense_layer_2(hidden_layer)
        hidden_layer = self.dense_layer_3(hidden_layer)

        # intermediate output to CFF prediction:
        cross_section_output = self.cross_section_output(hidden_layer)

        return {
            "cross-section": cross_section_output
        }

Fitting routine:

In [ ]:
class SurrogateCrossSectionFittingRoutine(tf.keras.Model):

    def __init__(self):
        super().__init__()

        self.base_model = CrossSectionSurrogateModel()

    def call(self, inputs, training = False):
        return self.base_model(inputs, training = training)

    def train_step(self, data):
        X_batch_data, y_batch_data = data

        with tf.GradientTape() as tape:
            predictions = self(X_batch_data, training = True)
            data_loss = cross_section_loss(y_batch_data, predictions)

        gradients = tape.gradient(data_loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.base_model.trainable_variables))

        return {
            "loss": data_loss,
        }

    def test_step(self, data):
        X_batch_data, y_batch_data = data
        predictions = self(X_batch_data, training = False)
        data_loss = cross_section_loss(y_batch_data, predictions)

        return {
            "loss": data_loss
        }

In [ ]:
NUMBER_OF_EPOCHS = 750

tf.keras.backend.clear_session()

dnn_model = SurrogateCrossSectionFittingRoutine()
dnn_model.compile(optimizer = tf.keras.optimizers.Adam())

dnn_model_history = dnn_model.fit(
    x_training, y_training,
    validation_data = (x_validation, y_validation),
    epochs = 1000, batch_size = len(x_training),
    verbose = 1)

number_of_epochs_run = len(dnn_model_history.epoch)
print(f"The model ran for {number_of_epochs_run} epochs before early stopping.")

In [ ]:
model_testing_loss = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
print(f"[INFO]: Test Loss: {model_testing_loss}")

In [ ]:
figure, axis = plt.subplots(1, 1, figsize = (6, 6))

axis.plot(dnn_model_history.history['loss'], 
    label = "Training Loss", color = 'orange', alpha = 0.6)
axis.plot(dnn_model_history.history['val_loss'], 
    label = "Validation Loss", color = 'purple', alpha = 0.6)

axis.set_xlabel(r"$\phi$ (radians)", fontsize = 14.)
axis.set_ylabel(r"$d^{4}\sigma$", fontsize = 14.)
axis.set_title(rf"Cross-Section Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
    fontsize = 14.)
axis.legend(fontsize = 14.)
axis.grid(visible = False)

axis.text(
    0.00, -0.11,
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = axis.transAxes,
    verticalalignment = 'top',
    horizontalalignment = 'left',
    fontsize = 9.,
)

for extension in ['png', 'eps']:
    figure.savefig(
        fname = f"./plots/xsec_surrogate_test_v2.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(figure)

In [ ]:
predictions = dnn_model.predict(x_data)["cross-section"].flatten()
actual = test_dataframe["unp_beam_unp_target_xsec"].values
phi_values = test_dataframe["phi"].values
residuals = actual - predictions

sum_squared_residuals = np.sum(residuals**2)
reduced_chi = sum_squared_residuals / len(phi_values)

residuals_figure, (ax1, ax2) = plt.subplots(2, 1, figsize = (10, 8), sharex = True, 
                               gridspec_kw = {'height_ratios': [3, 1]})

ax1.scatter(phi_values, actual, color = 'black', label = 'Experimental Data', alpha = 0.7)
ax1.plot(phi_values, predictions, color = 'red', lw = 2, label = 'DNN Prediction')
ax1.set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
ax1.set_ylabel(r"$d^{4}\sigma$", fontsize = 16.)
ax1.set_title(rf"Cross-Section Surrogate Model (Testing = ${model_testing_loss:.3f}$)", fontsize = 14.)
ax1.legend()
ax1.grid(True, linestyle = ':', alpha = 0.6)

# Bottom Panel: Residuals
ax2.scatter(phi_values, residuals, color = 'blue', alpha = 0.6)
ax2.axhline(0, color = 'black', linestyle = '--')
ax2.set_ylabel(rf'Residuals ($\chi^2_\nu = {reduced_chi:.3f}$)', fontsize = 16.)
ax2.set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
ax2.grid(True, linestyle = ':', alpha = 0.6)

for extension in ['png', 'eps']:
    residuals_figure.savefig(
        fname = f"./plots/xsec_surrogate_residuals_v2.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(residuals_figure)